# 10. Local Code Explainer

## What We're Building
A tool that takes code snippets and:
1. Explains what the code does in plain English
2. Identifies potential bugs or code smells
3. Suggests improvements
4. Generates inline documentation

**You will learn:**
- Code analysis with LLMs
- Multi-aspect evaluation from a single input
- Language-aware prompting
- Automated code review patterns

**Stack:** Ollama, LangChain

In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
print("LLM ready!")

## Step 1 — Code Explanation

In [ ]:
explain_prompt = ChatPromptTemplate.from_template(
    """You are an expert code teacher. Explain this code clearly.

```{language}
{code}
```

Provide:
1. **Overview**: What does this code do? (1-2 sentences)
2. **Step-by-step walkthrough**: Explain each important line/block
3. **Key concepts used**: List programming concepts demonstrated
4. **Example usage**: Show how to call/use this code

Explanation:"""
)

explain_chain = explain_prompt | llm | StrOutputParser()

# Sample code to explain
python_code = '''
def memoize(func):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# Calculate first 10 Fibonacci numbers
results = [fibonacci(i) for i in range(10)]
'''

explanation = explain_chain.invoke({"language": "python", "code": python_code})
print("=== CODE EXPLANATION ===")
print(explanation)

## Step 2 — Bug Detection and Code Smells

In [ ]:
review_prompt = ChatPromptTemplate.from_template(
    """You are a senior code reviewer. Analyze this code for issues.

```{language}
{code}
```

Check for:
1. **Bugs**: Logic errors, off-by-one errors, null/None issues
2. **Code smells**: Unnecessary complexity, poor naming, duplication
3. **Performance**: Inefficient patterns, memory issues
4. **Security**: Input validation, injection risks (if applicable)
5. **Best practices**: Style, conventions, documentation

For each issue found, explain:
- What the issue is
- Why it matters
- How to fix it

If the code is clean, say so and explain why it's well-written.

Code review:"""
)

review_chain = review_prompt | llm | StrOutputParser()

# Code with intentional issues
buggy_code = '''
def process_users(users):
    result = []
    for i in range(len(users)):
        user = users[i]
        if user["age"] > 0:
            name = user["name"].strip()
            if name != "":
                result.append({
                    "name": name,
                    "age": user["age"],
                    "category": "senior" if user["age"] > 65 else "adult" if user["age"] > 18 else "minor"
                })
    return result

def find_user(users, name):
    for user in users:
        if user["name"] == name:
            return user
    return None
'''

review = review_chain.invoke({"language": "python", "code": buggy_code})
print("=== CODE REVIEW ===")
print(review)

## Step 3 — Code Improvement Suggestions

In [ ]:
improve_prompt = ChatPromptTemplate.from_template(
    """You are an expert code refactorer. Improve this code.

```{language}
{code}
```

Provide:
1. The improved version of the code
2. A list of changes made and why each change improves the code

Keep the same functionality. Focus on:
- Readability and Pythonic idioms
- Performance where applicable
- Type hints and documentation
- Error handling at appropriate boundaries

Improved code:"""
)

improve_chain = improve_prompt | llm | StrOutputParser()
improvement = improve_chain.invoke({"language": "python", "code": buggy_code})
print("=== IMPROVED CODE ===")
print(improvement)

## Step 4 — Documentation Generator

In [ ]:
docstring_prompt = ChatPromptTemplate.from_template(
    """Add comprehensive docstrings to this code. Use Google-style docstrings.

```{language}
{code}
```

For each function/class, add:
- Brief description
- Args with types and descriptions
- Returns with type and description
- Raises (if applicable)
- Example usage in the docstring

Return the full code with docstrings added:"""
)

docstring_chain = docstring_prompt | llm | StrOutputParser()

undocumented_code = '''
def paginate(items, page, per_page):
    start = (page - 1) * per_page
    end = start + per_page
    total_pages = -(-len(items) // per_page)
    return {
        "data": items[start:end],
        "page": page,
        "per_page": per_page,
        "total": len(items),
        "total_pages": total_pages,
        "has_next": page < total_pages,
        "has_prev": page > 1,
    }
'''

documented = docstring_chain.invoke({"language": "python", "code": undocumented_code})
print("=== DOCUMENTED CODE ===")
print(documented)

In [ ]:
# Full code analysis pipeline
def analyze_code(code: str, language: str = "python") -> dict:
    """Run complete code analysis: explain, review, improve, document."""
    results = {}

    print("1/4: Explaining code...")
    results["explanation"] = explain_chain.invoke({"language": language, "code": code})

    print("2/4: Reviewing for issues...")
    results["review"] = review_chain.invoke({"language": language, "code": code})

    print("3/4: Suggesting improvements...")
    results["improvement"] = improve_chain.invoke({"language": language, "code": code})

    print("4/4: Generating documentation...")
    results["documented"] = docstring_chain.invoke({"language": language, "code": code})

    return results

# Test with any code snippet
sample = '''
def flatten(lst):
    out = []
    for item in lst:
        if isinstance(item, list):
            out.extend(flatten(item))
        else:
            out.append(item)
    return out
'''

analysis = analyze_code(sample)
for section, content in analysis.items():
    print(f"\n{'='*50}")
    print(f"{section.upper()}")
    print(f"{'='*50}")
    print(content)

## Summary & Next Steps

**What you learned:**
- ✅ Code analysis with LLMs (explanation, review, improvement)
- ✅ Multi-aspect evaluation pipelines
- ✅ Language-aware prompting
- ✅ Automated documentation generation

**Next steps:**
- Add support for multi-file analysis
- Build a code review agent (Project 93)
- Compare analysis quality across different models
- Add severity scoring for issues found

---
*This completes the Beginner Local LLM Apps group (Projects 1-10)!
Next: Projects 11-20 explore Local RAG applications.*

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
